# Penjelasan kode · Diagnosis Kerusakan Bearing (Tim Bearing)

<a href="https://colab.research.google.com/github/RizkyMuhammad23/bearing-diagnosis/blob/main/notebooks/penjelasan_kode_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook ini dipakai di video presentasi untuk menjelaskan kode. Pembicara tidak bergantian: Rizky menjelaskan bagian 1–4 sampai selesai, lalu Irham bagian 5–7. Setiap bagian berisi kode yang
dijalankan dan kalimat yang diucapkan.

| Bagian | Pembicara | Waktu di video |
|---|---|---|
| 1–4: data, pembersihan, fitur, model, evaluasi, lalu hasil dan keterbatasan | Rizky Muhammad Ikhsan (2403329) | 00:25–02:20 |
| 5–7: artefak, validasi masukan, API, aplikasi | Muhammad Khairul Irham (2402023) | 02:20–03:10 |

**Sebelum merekam:** pilih *Runtime → Run all* dan tunggu sampai semua sel selesai (sekitar 2 menit). Saat merekam,
cukup gulir dan tunjuk sel yang sedang dijelaskan. Tidak perlu menjalankan ulang.

Sumber kode: https://github.com/RizkyMuhammad23/bearing-diagnosis

## 0. Persiapan (tidak perlu dijelaskan di video)

In [ ]:
# Ambil repo dari GitHub (di Colab), lalu pasang paket untuk API.
import os, sys
if not os.path.exists("src/features.py"):
    if os.path.exists("../src/features.py"):
        os.chdir("..")                      # dijalankan dari folder notebooks/ di komputer sendiri
    else:
        if not os.path.exists("bearing-diagnosis"):
            !git clone -q --depth 1 https://github.com/RizkyMuhammad23/bearing-diagnosis
        os.chdir("bearing-diagnosis")
%pip install -q fastapi uvicorn requests
%pip install -q scikit-learn==1.9.1   # versi yang sama dengan saat model dilatih, agar angka sama dengan laporan
sys.path.insert(0, os.getcwd())

import inspect, json, warnings
import numpy as np, pandas as pd, sklearn, joblib
from IPython.display import Code, display
warnings.filterwarnings("ignore")
lihat = lambda fungsi: display(Code(inspect.getsource(fungsi), language="python"))  # tampilkan kode sebuah fungsi
print("folder:", os.getcwd(), "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)
if sklearn.__version__ != "1.9.1":
    print("PERINGATAN: scikit-learn bukan 1.9.1, angka bisa sedikit berbeda dari laporan. Pilih Runtime > Restart session lalu Run all.")

## 1. Data mentah dan masalahnya · *Rizky*

In [ ]:
df_raw = pd.read_csv("data/raw.csv")
print(df_raw.shape)
pd.Series({
    "duplikat persis": df_raw.duplicated().sum(),
    "variasi tulisan label": df_raw["kondisi"].nunique(),
    "kurtosis kosong": df_raw["kurtosis"].isna().sum(),
    "suhu kosong": df_raw["suhu_housing_C"].isna().sum(),
    "kecepatan_rpm = 0": (df_raw["kecepatan_rpm"] == 0).sum(),
    "rms_g negatif": (df_raw["rms_g"] < 0).sum(),
})

> **Diucapkan (Rizky):** *"Data klien berisi 15.220 rekaman getaran dengan enam masalah: duplikat, label yang ditulis dalam 16 variasi, kurtosis dan suhu kosong, kecepatan putar nol, dan RMS getaran negatif."*

## 2. Pembersihan data: `clean_raw()` di `src/features.py` · *Rizky*

In [ ]:
from src.features import clean_raw, add_features, LABEL_MAP, CLASSES, RAW_FEATURES, NUM_COLS, ENGINEERED
lihat(clean_raw)

In [ ]:
df, log = clean_raw(df_raw)
print(json.dumps(log, indent=2))
df["kondisi"].value_counts()

> **Diucapkan (Rizky):** *"Fungsi clean_raw menghapus duplikat, menyeragamkan label ke empat kelas, mengambil nilai mutlak RMS negatif karena puncak dibagi RMS tetap cocok dengan crest factor, dan menjadikan kecepatan nol sebagai nilai kosong. Nilai kosong diisi median di dalam pipeline, jadi tidak ada informasi yang bocor dari data test."*

## 3. Fitur turunan: `add_features()` · *Rizky*

In [ ]:
lihat(add_features)

In [ ]:
fe = add_features(df[RAW_FEATURES]).assign(kondisi=df["kondisi"])
fe.groupby("kondisi")[["orde_frekuensi", "porsi_bpfo", "porsi_bpfi", "porsi_bsf"]].median().loc[CLASSES].round(2)

> **Diucapkan (Rizky):** *"Fitur terpenting adalah orde frekuensi, yaitu frekuensi dominan dibagi frekuensi putar poros. Porsi energi juga menunjukkan polanya: cincin luar menonjol di BPFO, cincin dalam di BPFI, dan bola di BSF."*

## 4. Model, pemilihan, dan evaluasi · *Rizky*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, confusion_matrix
from src.features import make_preprocessor, feature_step

X, y = df[RAW_FEATURES], df["kondisi"]
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

baseline = Pipeline([("prep", make_preprocessor(NUM_COLS, scale=True)),
                     ("clf", LogisticRegression(max_iter=2000))])
final = Pipeline([("fe", feature_step()),                                   # add_features()
                  ("prep", make_preprocessor(NUM_COLS + ENGINEERED, scale=False)),   # imputasi median + one-hot
                  ("clf", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=800, min_samples_leaf=40,
                                                         l2_regularization=1.0, random_state=42))])
for nama, m in [("baseline regresi logistik", baseline), ("HistGradientBoosting final", final)]:
    m.fit(X_train, y_train)
    print(f"{nama:28s} macro F1 validation = {f1_score(y_val, m.predict(X_val), average='macro'):.4f}")

In [ ]:
# Ambang keyakinan: nilai terkecil yang membuat tebakan yang lolos ambang benar >= 90% (data validation).
proba = final.predict_proba(X_val); tebak = final.classes_[proba.argmax(1)]; yakin = proba.max(1)
for t in [0.55, 0.60, 0.65, 0.70]:
    lolos = yakin >= t
    print(f"ambang {t:.2f}: {lolos.mean():.1%} lolos, akurasinya {(tebak[lolos] == y_val.values[lolos]).mean():.3f}")

In [ ]:
# Data test dibuka SEKALI, setelah model dan ambang dipilih. Model dilatih ulang dengan train + validation.
X_trval, y_trval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
for m in (baseline, final): m.fit(X_trval, y_trval)
print(f"macro F1 test: baseline {f1_score(y_test, baseline.predict(X_test), average='macro'):.4f} | "
      f"final {f1_score(y_test, final.predict(X_test), average='macro'):.4f}")
pd.DataFrame(confusion_matrix(y_test, final.predict(X_test), labels=CLASSES), index=CLASSES, columns=CLASSES)

> **Diucapkan (Rizky):** *"Data dibagi stratified 70, 15, dan 15 persen. Dalam satu pipeline, HistGradientBoosting dengan fitur turunan mengalahkan baseline regresi logistik di data validation. Ambang keyakinan 0,65 dipilih agar tebakan di atasnya benar minimal 90 persen. Data test dibuka sekali di akhir: macro F1 naik dari 0,747 ke 0,799, dan kesalahan terbanyak adalah bearing rusak yang ditebak normal."*

> **Diucapkan (Rizky):** *"Dari confusion matrix, antarjenis kerusakan jarang tertukar. Yang paling sering terjadi adalah bearing rusak yang frekuensi dominannya masih di satu atau dua kali putaran, sehingga mirip rekaman normal. Karena itu rekaman dengan keyakinan di bawah 0,65 diserahkan ke analis. Keterbatasannya, data ini sintetis, hanya untuk satu jenis bearing dengan kecepatan 1.200 sampai 3.000 rpm, dan tidak mencakup kerusakan ganda. Selanjutnya Irham akan menjelaskan bagaimana model ini dipakai oleh API dan aplikasi."*

## 5. Artefak model: `artifacts/` · *Irham*

In [ ]:
meta = json.load(open("artifacts/metadata.json"))
try:
    model = joblib.load("artifacts/model.joblib")
    model.predict(X_test.head(1))
    sumber = "artifacts/model.joblib dari repo"
except Exception as e:
    # Versi scikit-learn di Colab berbeda dengan saat model disimpan: simpan ulang model yang baru dilatih di atas.
    # Ini hanya mengubah salinan repo di Colab, bukan repo di GitHub.
    joblib.dump(final, "artifacts/model.joblib")
    model, sumber = final, "model yang baru dilatih di notebook ini (versi scikit-learn berbeda)"
print("model dimuat dari:", sumber)
print(json.dumps({k: meta[k] for k in ["versi_model", "algoritma", "kelas", "threshold"]}, indent=2, ensure_ascii=False))
print(json.dumps(meta["metrik"]["final"], indent=2))

> **Diucapkan (Irham):** *"Pipeline lengkap disimpan sebagai model.joblib, ditemani metadata.json berisi versi, fitur masukan, ambang keyakinan, dan metrik. File inilah yang dimuat oleh API dan aplikasi."*

## 6. Validasi masukan dan prediksi: `src/predict.py` · *Irham*

In [ ]:
from src import predict as P
P.load_model.cache_clear()
contoh = json.load(open("data/contoh_masukan.json"))
for nama in ["outer_race", "aneh_tapi_diproses", "mustahil_ditolak"]:
    try:
        h = P.prediksi(contoh[nama])
        print(f"{nama:20s} -> {h['kondisi']} ({h['keyakinan']:.0%}), perlu dicek analis: {h['perlu_cek_analis']}")
        for p in h["peringatan"]: print("    catatan:", p)
    except P.MasukanTidakValid as e:
        print(f"{nama:20s} -> DITOLAK:", e.pesan)

In [ ]:
# Bagian validasi yang memisahkan masukan mustahil (ditolak) dan masukan aneh (diproses dengan catatan)
src = inspect.getsource(P.validasi)
display(Code(src[src.index("    # Kolom angka"):src.index("    # Pemeriksaan antar kolom")], language="python"))

> **Diucapkan (Irham):** *"Setiap masukan divalidasi dulu. Masukan mustahil, misalnya energi negatif, ditolak dengan alasan yang jelas. Masukan aneh, misalnya RMS negatif atau kecepatan nol, tetap diprediksi tetapi diberi catatan dan ditandai perlu dicek analis."*

## 7. API FastAPI dan aplikasi Streamlit · *Irham*

In [ ]:
# Jalankan API sungguhan di latar belakang (uvicorn), lalu kirim permintaan HTTP.
import subprocess, time, requests
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.api.main:app", "--port", "8000"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        print("GET  /health  ->", requests.get("http://127.0.0.1:8000/health").json()); break
    except requests.ConnectionError:
        time.sleep(0.5)
r = requests.post("http://127.0.0.1:8000/predict", json=contoh["inner_race"])
print("POST /predict ->", r.status_code, r.json()["kondisi"], r.json()["probabilitas"])
r = requests.post("http://127.0.0.1:8000/predict", json=contoh["mustahil_ditolak"])
print("POST /predict ->", r.status_code, r.json()["detail"])

In [ ]:
# Inti app/ui/streamlit_app.py: muat halaman BearingCek (berisi model dari artifacts/) dan tampilkan.
kode = open("app/ui/streamlit_app.py").read()
display(Code(kode[kode.index("@st.cache_resource"):], language="python"))
print("Aplikasi online: https://bearing-diagnosis-mkb.streamlit.app")

> **Diucapkan (Irham):** *"API FastAPI menjawab health dan predict. Masukan valid mendapat kondisi dan probabilitas, sedangkan masukan mustahil mendapat kode 422. Aplikasi Streamlit BearingCek memuat model yang sama dari folder artifacts dan sudah online. Sekarang saya demokan."*